# 03 · Comparaison des modèles

On lit les scores écrits par `python train.py` dans `models/metrics.json`.
Méthode : validation croisée (3 plis) pour **choisir** le meilleur modèle, puis **un seul** passage sur le jeu de test (20 %, jamais utilisé pour choisir).

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, "..")                    # pour importer les modules de ml/
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

DATA = Path("../dataset")                   # dossier contenant styles.csv et images/

import json
metrics = json.loads((Path("../models") / "metrics.json").read_text())
print("train :", metrics["n_train"], "images | test :", metrics["n_test"], "images")

## 1. Validation croisée : tous les modèles, toutes les cibles

In [ ]:
cv = pd.DataFrame({t: {m: v["f1_macro"] for m, v in d["cv"].items()} for t, d in metrics["targets"].items()})
display(cv.round(3))
cv.plot.bar(figsize=(9, 4), rot=0, ylabel="F1 macro (validation croisée)", ylim=(0, 1)); plt.show()
print("Meilleur modèle par cible :", {t: d["best"] for t, d in metrics["targets"].items()})

## 2. Scores sur le jeu de test (modèle retenu)

In [ ]:
rows = {t: {k: d["test"][k] for k in ("accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted")}
        for t, d in metrics["targets"].items()}
display(pd.DataFrame(rows).T.round(3))

In [ ]:
# disponible si train.py a été lancé avec --full-compare
for t, d in metrics["targets"].items():
    if "test_all" in d:
        print(f"\nCible : {t}"); display(pd.DataFrame(d["test_all"]).T.round(3))

## 3. Matrice de confusion : quels vêtements sont confondus ?

In [ ]:
target = "type"
t = metrics["targets"][target]["test"]
cm, classes = np.array(t["confusion_matrix"], dtype=float), t["classes"]
norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(classes)), classes, rotation=60, ha="right"); ax.set_yticks(range(len(classes)), classes)
ax.set_xlabel("Prédit"); ax.set_ylabel("Réel"); ax.set_title(f"Matrice de confusion normalisée : {target}")
for i in range(len(classes)):
    for j in range(len(classes)):
        if cm[i, j]: ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=7, color="white" if norm[i, j] > .5 else "black")
plt.colorbar(im); plt.show()

In [ ]:
report = pd.DataFrame(t["per_class"]).T.drop(["accuracy", "macro avg", "weighted avg"], errors="ignore")
display(report.sort_values("f1-score").round(2))   # classes les plus difficiles en premier

## 4. Tester sur une de vos photos

In [ ]:
import predict
PHOTO = "chemin/vers/ma_photo.jpg"      # un seul vêtement, centré, fond uni
from PIL import Image
if Path(PHOTO).exists():
    plt.imshow(Image.open(PHOTO)); plt.axis("off"); plt.show()
    print(json.dumps(predict.predict_image(PHOTO), ensure_ascii=False, indent=2))
else:
    print("Modifiez PHOTO avec le chemin d'une vraie photo.")

## Pour le rapport
- Justifier le **F1 macro** : les classes sont déséquilibrées, l'accuracy seule serait trompeuse.
- Commenter les **confusions** de la matrice (ex. Pantalon ↔ Jogging) par la ressemblance visuelle.
- Expliquer pourquoi la **couleur** est plus facile (caractéristiques adaptées, calculées sur le vêtement seulement) que le **type**.
